# FaceProof — hardening checks (council-review follow-ups)

Three cheap checks that close the last open review items. **GPU on.** Does **not** touch the frozen
`results.csv` — writes a separate `reports/hardening_checks.md` on Drive.

1. **DCT leakage check, real vs T2I** — the Day-2 leakage gate was only ever run real-vs-StyleGAN.
   Here we run the same trivial DCT+SVM probe on held-out reals vs each T2I generator's
   compression-matched crops. Near-chance = the below-chance collapse is not a frequency/compression
   shortcut. High accuracy = a preprocessing confound contributes — report it honestly either way.
2. **Bootstrap 95% CIs on the below-chance T2I cells** — the headline numbers were single-point.
   CIs let us say per-cell whether a value is genuinely *below* chance (CI entirely under 0.5) or
   only *at* chance (CI crosses 0.5). C4/C1 on all three generators + C2 on DALL·E 3 (the deepest).
3. **CLIP temperature json** — refit temperature on the validation split and save
   `models/probes/c4_clip_temperature.json` so the demo's calibrated confidence is real.

At the end, copy the printed SUMMARY block back into the chat so it can be folded into the report.

## 1. Setup (GPU on) + data

In [ ]:
REPO_URL = "https://github.com/Ezed9/faceProof.git"
BRANCH   = "day11-14-completion"   # this notebook needs the fixed bootstrap_ci; set to "main" after merge
!git clone -b $BRANCH $REPO_URL
%cd faceProof
!pip install -q kaggle open_clip_torch scikit-learn joblib pyyaml
import sys, os
sys.path.append(os.getcwd())

In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np, pandas as pd
drive.mount('/content/drive')
ROOT          = Path('/content/drive/MyDrive/faceproof')
CROPS_ROOT    = ROOT / 'data' / 'crops'
FEATURES_ROOT = ROOT / 'models' / 'features'
PROBES_ROOT   = ROOT / 'models' / 'probes'
REPORTS_ROOT  = ROOT / 'reports'
MANIFEST      = ROOT / 'data' / 'manifest.csv'
REPORTS_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY = {}   # collected results -> reports/hardening_checks.md

In [ ]:
import glob
from google.colab import files
def find_t2i_csv():
    cs = [c for c in glob.glob('/content/**/*.csv', recursive=True) if 't2i' in c.lower() or 'sfhq' in c.lower()]
    return cs[0] if cs else None
if find_t2i_csv() is None:
    print('Upload kaggle.json:'); files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d selfishgene/sfhq-t2i-synthetic-faces-from-text-2-image-models -p /content/t2i --unzip
CSV = find_t2i_csv(); assert CSV, 'T2I CSV not found'

meta = pd.read_csv(CSV)
COL_IMAGE, COL_GEN = 'image_filename', 'model_used'
img_root = Path(CSV).parent
index = {q.name: str(q) for q in img_root.rglob('*') if q.suffix.lower() in {'.jpg','.jpeg','.png'}}
meta['path'] = meta[COL_IMAGE].map(index.get)
meta = meta.dropna(subset=['path'])
meta = meta[meta[COL_GEN].isin(['SDXL','FLUX1_schnell','DALLE3'])].copy()
print(meta[COL_GEN].value_counts())

In [ ]:
import yaml, joblib, shutil
from PIL import Image
from src.preprocessing import preprocess_image

dcfg = yaml.safe_load(open('configs/data.yaml')); IMG, Q = dcfg['image_size'], dcfg['jpeg_match_quality']
mcfg = yaml.safe_load(open('configs/model.yaml'))['clip']

_mani = pd.read_csv(MANIFEST)
real_rows  = _mani[(_mani.label==0) & (_mani['split']=='test_indist')]
real_paths = real_rows['path'].tolist()[:3000]
assert real_paths, 'no held-out reals in manifest'

def crops_for(paths, tag, limit=3000):
    """Compression-match fakes through the standard 224/JPEG-90 pipeline (same as nb05/nb08)."""
    out = Path(f'/content/_hard_{tag}')
    if out.exists(): shutil.rmtree(out)
    out.mkdir(parents=True)
    kept=[]
    for i,p in enumerate(paths[:limit]):
        im = preprocess_image(p, IMG, Q, detector=None)
        if im is None: continue
        op = out/f'{i:06d}.jpg'; im.save(op,'JPEG',quality=Q); kept.append(str(op))
    return kept

# Preprocess each generator's fakes ONCE; reused by sections 2 and 3.
FAKES = {gen: crops_for(grp['path'].tolist(), gen) for gen, grp in meta.groupby(COL_GEN)}
print({g: len(v) for g,v in FAKES.items()})

## 2. Check 1 — DCT leakage, real vs T2I

Same gate as Day 2 (`src.leakage_check.run_leakage_check`, 5-fold CV, warn at >0.80), but real-vs-T2I.
Reals are held-out crops (already preprocessed); fakes are the compression-matched crops from above.

In [ ]:
from src.leakage_check import run_leakage_check

# Stage 500 held-out real crops in a temp folder (run_leakage_check is folder-based).
leak_real = Path('/content/_leak_real')
if leak_real.exists(): shutil.rmtree(leak_real)
leak_real.mkdir(parents=True)
for i,p in enumerate(real_paths[:500]):
    shutil.copy(p, leak_real/f'{i:06d}.jpg')

SUMMARY['leakage_t2i'] = {}
for gen, fake_crops in FAKES.items():
    leak_fake = Path(f'/content/_leak_{gen}')
    if leak_fake.exists(): shutil.rmtree(leak_fake)
    leak_fake.mkdir(parents=True)
    for i,p in enumerate(fake_crops[:500]):
        shutil.copy(p, leak_fake/f'{i:06d}.jpg')
    print(f'--- real vs {gen} ---')
    acc = run_leakage_check(leak_real, leak_fake, n_per_class=500)
    SUMMARY['leakage_t2i'][gen] = round(float(acc), 3)

## 3. Check 2 — bootstrap 95% CIs on the below-chance T2I cells

C4 (CLIP) + C1 (ResNet) saved probes vs held-out reals per generator; C2 (fine-tuned ResNet) on
DALL·E 3 (the deepest cell, point 0.170). CI entirely below 0.5 = genuinely below chance;
CI crossing 0.5 = report as "at chance".

In [ ]:
from src.features import extract_clip, extract_resnet
from src.probe import predict_proba
from src.metrics import bootstrap_ci

probes = {c: joblib.load(PROBES_ROOT / f'{c}.joblib') for c in ['c1_resnet','c4_clip']}

Xr_clip = extract_clip(real_paths, backbone=mcfg['backbone'], pretrained=mcfg['pretrained'], batch_size=64, cache=None)
Xr_res  = extract_resnet(real_paths, batch_size=64, cache=None)

SUMMARY['t2i_ci'] = []
for gen, fake_crops in FAKES.items():
    Xf_clip = extract_clip(fake_crops, backbone=mcfg['backbone'], pretrained=mcfg['pretrained'], batch_size=64, cache=None)
    Xf_res  = extract_resnet(fake_crops, batch_size=64, cache=None)
    y = np.r_[np.zeros(len(real_paths)), np.ones(len(fake_crops))]
    for cond, (Xr, Xf) in {'c4_clip': (Xr_clip, Xf_clip), 'c1_resnet': (Xr_res, Xf_res)}.items():
        s = predict_proba(probes[cond], np.vstack([Xr, Xf]))
        point, lo, hi = bootstrap_ci(y, s, n=2000, seed=13)
        verdict = 'below chance' if hi < 0.5 else ('above chance' if lo > 0.5 else 'AT chance (CI crosses 0.5)')
        SUMMARY['t2i_ci'].append({'cond':cond,'gen':gen,'AUROC':round(point,4),'lo':round(lo,4),'hi':round(hi,4),'verdict':verdict})
        print(f'{cond} {gen}: {point:.4f} [{lo:.4f},{hi:.4f}] -> {verdict}')

In [ ]:
# C2 fine-tuned ResNet on DALL-E 3 (deepest single-point cell: 0.170).
import torch, torch.nn as nn
import torchvision.transforms as T
from torchvision.models import resnet50

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
tf  = T.Compose([T.Resize((224,224)), T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
c2  = resnet50(); c2.fc = nn.Linear(c2.fc.in_features, 1)
c2.load_state_dict(torch.load(PROBES_ROOT/'c2_resnet_finetune.pt', map_location=DEV))
c2 = c2.to(DEV).eval()

@torch.no_grad()
def c2_scores(paths, bs=64):
    out=[]
    for i in range(0, len(paths), bs):
        xb = torch.stack([tf(Image.open(p).convert('RGB')) for p in paths[i:i+bs]]).to(DEV)
        out.append(torch.sigmoid(c2(xb).squeeze(1)).cpu().numpy())
    return np.concatenate(out)

dalle = FAKES['DALLE3']
y  = np.r_[np.zeros(len(real_paths)), np.ones(len(dalle))]
s  = c2_scores(list(real_paths) + list(dalle))
point, lo, hi = bootstrap_ci(y, s, n=2000, seed=13)
verdict = 'below chance' if hi < 0.5 else ('above chance' if lo > 0.5 else 'AT chance (CI crosses 0.5)')
SUMMARY['t2i_ci'].append({'cond':'c2_resnet_finetune','gen':'DALLE3','AUROC':round(point,4),'lo':round(lo,4),'hi':round(hi,4),'verdict':verdict})
print(f'c2 DALLE3: {point:.4f} [{lo:.4f},{hi:.4f}] -> {verdict}')

## 4. Check 3 — CLIP temperature json (for the demo's calibrated confidence)

Refit temperature on the **validation** split (same as Day 6) and save it where `demo/app.py` looks:
`models/probes/c4_clip_temperature.json`.

In [ ]:
import json as _json
from src.calibration import fit_temperature, apply_temperature
from src.metrics import expected_calibration_error

df = pd.read_csv(MANIFEST)
Xc = np.load(FEATURES_ROOT/'clip_all.npy')
assert len(Xc) == len(df), f'features {len(Xc)} != manifest {len(df)} - re-run nb02 or check cache name'
val = (df['split']=='val').values
yv  = df['label'].values[val]

logits = probes['c4_clip'].decision_function(Xc[val])
T = fit_temperature(logits, yv)
ece_before = expected_calibration_error(yv, 1/(1+np.exp(-logits)))
ece_after  = expected_calibration_error(yv, apply_temperature(logits, T))

tmp_path = PROBES_ROOT/'c4_clip_temperature.json'
tmp_path.write_text(_json.dumps({'T': round(float(T), 4)}))
SUMMARY['temperature'] = {'T': round(float(T),4), 'val_ECE_before': round(float(ece_before),4), 'val_ECE_after': round(float(ece_after),4)}
print(f'CLIP temperature T={T:.4f}  (val ECE {ece_before:.4f} -> {ece_after:.4f})  saved -> {tmp_path}')

## 5. Summary — save to Drive + PASTE THIS BLOCK BACK

In [ ]:
lines = ['# Hardening checks — results', '',
         '## 1. DCT leakage, held-out real vs T2I (5-fold CV accuracy; pass = near 0.5, warn > 0.80)']
for gen, acc in SUMMARY['leakage_t2i'].items():
    lines.append(f'- real vs {gen}: **{acc}**')
lines += ['', '## 2. Bootstrap 95% CIs on T2I cells (2000 resamples, seed 13)',
          '| condition | generator | AUROC | 95% CI | verdict |', '|---|---|---|---|---|']
for r in SUMMARY['t2i_ci']:
    lines.append(f"| {r['cond']} | {r['gen']} | {r['AUROC']} | [{r['lo']}, {r['hi']}] | {r['verdict']} |")
t = SUMMARY['temperature']
lines += ['', '## 3. CLIP temperature (val-fit, saved to models/probes/c4_clip_temperature.json)',
          f"- T = **{t['T']}** (val ECE {t['val_ECE_before']} -> {t['val_ECE_after']})"]
out = REPORTS_ROOT/'hardening_checks.md'
out.write_text('\n'.join(lines) + '\n')
print('saved', out, '\n')
print('='*20, 'SUMMARY — paste everything below back into the chat', '='*20)
print('\n'.join(lines))